# Zonal: weighted sums, block collation, coarse-cell weights, hierarchy roll-up, facilities

Synthetic block, four quadrant zones, six facilities. The NaN policy is a required argument;
see `docs/conventions.md`.

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from affine import Affine
from shapely.geometry import Point, box

from idd_spatial_analysis.constants import WORLD_CYLINDRICAL_EQUAL_AREA
from idd_spatial_analysis.frame import BlockGrid
from idd_spatial_analysis.raster import rasterize_zones
from idd_spatial_analysis.zonal import (
    collate_blocks,
    nearest_facility_distance,
    rollup_hierarchy,
    share_within,
    weighted_sums,
    zone_weights,
)

In [ ]:
grid = BlockGrid(
    crs=WORLD_CYLINDRICAL_EQUAL_AREA,
    transform=Affine(100.0, 0, 0, 0, -100.0, 6400.0),
    height=64,
    width=64,
)
rng = np.random.default_rng(0)
population = rng.gamma(2.0, 50.0, size=grid.shape).astype("float32")
population[:8, :8] = np.nan
temperature = (25 + 5 * rng.standard_normal(grid.shape)).astype("float32")
temperature[10:20, 40:50] = np.nan  # a hole in the value field inside zone 2
zones = gpd.GeoDataFrame(
    {"zone": np.array([1, 2, 3, 4], dtype="int32")},
    geometry=[
        box(0, 3200, 3200, 6400),
        box(3200, 3200, 6400, 6400),
        box(0, 0, 3200, 3200),
        box(3200, 0, 6400, 3200),
    ],
    crs=grid.crs,
)
mask, windows = rasterize_zones(zones, "zone", grid)

In [ ]:
# Population-weighted sums per zone under both NaN policies (they differ only in zone 2).
keep = weighted_sums(mask, population, temperature, nan_policy="keep_weight")
drop = weighted_sums(mask, population, temperature, nan_policy="drop_pixel")
print(keep.assign(mean=keep["weighted_sum"] / keep["weight_sum"]))
print(drop.assign(mean=drop["weighted_sum"] / drop["weight_sum"]))

In [ ]:
# A stack of slices (days, say) gets a slice column; blocks are collated by adding partial sums.
stack = np.stack([temperature, temperature + 1])
by_slice = weighted_sums(mask, population, stack, nan_policy="drop_pixel")
left = weighted_sums(mask[:, :40], population[:, :40], temperature[:, :40], nan_policy="drop_pixel")
right = weighted_sums(
    mask[:, 40:], population[:, 40:], temperature[:, 40:], nan_policy="drop_pixel"
)
print(by_slice.head(8))
collated = collate_blocks([left, right])
print(
    np.allclose(collated[["weighted_sum", "weight_sum"]], drop[["weighted_sum", "weight_sum"]]),
    "<- collated partials equal the whole-block sums",
)

In [ ]:
# Sparse weights from a coarse (climate) grid to zones: apply them to any coarse field later.
coarse = BlockGrid(
    crs=grid.crs, transform=Affine(1600.0, 0, 0, 0, -1600.0, 6400.0), height=4, width=4
)
weights = zone_weights(mask, population, grid, coarse)
print(weights.head())
print(weights.groupby("zone")["weight"].sum())

In [ ]:
# Roll most-detailed sums up a hierarchy: sum first, divide last.
hierarchy = pd.DataFrame(
    {
        "location_id": [10, 11, 12, 1, 2, 3, 4],
        "parent_id": [0, 10, 10, 11, 11, 12, 12],
        "level": [0, 1, 1, 2, 2, 2, 2],
    }
)
print(rollup_hierarchy(drop, hierarchy))

In [ ]:
# Nearest facility per pixel, any facility or only those on the same zone (zone 3 has none).
facilities = gpd.GeoDataFrame(
    {"facility_id": np.arange(6, dtype="int32")},
    geometry=[
        Point(x, y)
        for x, y in [
            (500, 6000),
            (2500, 4000),
            (5000, 5000),
            (4000, 1000),
            (6000, 3000),
            (3300, 300),
        ]
    ],
    crs=grid.crs,
)
any_facility = nearest_facility_distance(mask, facilities, grid, within_zone=False)
same_zone = nearest_facility_distance(mask, facilities, grid, within_zone=True, windows=windows)
print(np.nanmax(any_facility), np.nanmax(same_zone), np.isnan(same_zone[mask == 3]).all())

In [ ]:
# Share of population within thresholds of a facility, per zone.
print(share_within(same_zone, population, mask, [1000.0, 3000.0]))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 4))
for ax, field, title in (
    (axes[0], any_facility, "distance to any facility (m)"),
    (axes[1], same_zone, "same-zone facility (m)"),
):
    image = ax.imshow(field, extent=(0, 6400, 0, 6400))
    ax.scatter(facilities.geometry.x, facilities.geometry.y, c="red", s=12)
    ax.set_title(title)
    fig.colorbar(image, ax=ax)
plt.show()